# EuroAI Practice Round: Micro-Grid Fault Localization

Welcome to the starter notebook for the Micro-Grid Fault Localization task. This notebook provides a clean workflow to load the datasets, inspect structural anomalies, engineer basic features, train a baseline classifier, and format your final submission.

### Pipeline Overview:
1. **Exploratory Data Analysis (EDA)**: Inspect shape, data types, and target distributions.
2. **Feature Engineering**: Compute domain-specific voltage deltas and ratio interaction indicators.
3. **Validation Strategy**: Implement a Stratified K-Fold cross-validation scheme to track ROC-AUC.
4. **Baseline Modeling**: Train a Random Forest estimator.
5. **Submission Generation**: Output a clean `submission.csv` mapping predictions to test node identifiers.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

print("Libraries successfully imported!")

## 1. Load Data
Let's check the contents of `train.csv` and `test.csv` to understand what features we have access to.

In [ ]:
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

print(f"Train Data Shape: {train_df.shape}")
print(f"Test Data Shape:  {test_df.shape}")
train_df.head()

## 2. Target Variable Exploration
Let's look at the class balance inside our ground-truth configurations.

In [ ]:
target_counts = train_df['fault_status'].value_counts(normalize=True)
print("Target Distribution:")
print(target_counts)

train_df['fault_status'].value_counts().plot(kind='bar', color=['#4F81BD', '#C0504D'])
plt.title('Fault Status Balance (0 = Nominal, 1 = Fault)')
plt.ylabel('Count')
plt.show()

## 3. Feature Engineering
Olympiad style tabular problems usually reward smart manual domain interactions over complex brute-force modeling. Let's build a clean function to generate some key mathematical features from the alpha and beta voltage readings.

In [ ]:
def engineer_features(df):
    # Copy to isolate mutations
    data = df.copy()
    
    # 1. Capture absolute voltage drop between stages
    data['voltage_delta'] = data['stage_alpha'] - data['stage_beta']
    
    # 2. Voltage efficiency ratio (safely handle division by zero)
    data['efficiency_ratio'] = data['stage_beta'] / (data['stage_alpha'] + 1e-6)
    
    # 3. Structural physical interaction: drop amplified by line resonance factor
    data['drop_vs_harmonic'] = data['voltage_delta'] * data['harmonic_index']
    
    return data

train_feat = engineer_features(train_df)
test_feat = engineer_features(test_df)

print(f"Engineered Train Shape: {train_feat.shape}")
train_feat.head()

## 4. Cross-Validation and Model Training
We use standard `StratifiedKFold` to ensure stable metric evaluations across our limited training split.

In [ ]:
features = ['stage_alpha', 'stage_beta', 'harmonic_index', 'voltage_delta', 'efficiency_ratio', 'drop_vs_harmonic']
X = train_feat[features]
y = train_feat['fault_status']
X_test = test_feat[features]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(train_feat))
test_preds = np.zeros(len(test_feat))
cv_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    # Regularized Random Forest optimized for smaller robust profiles
    model = RandomForestClassifier(n_estimators=100, max_depth=4, random_state=42)
    model.fit(X_train, y_train)
    
    # Evaluate Validation Split
    val_prob = model.predict_proba(X_val)[:, 1]
    oof_preds[val_idx] = val_prob
    fold_auc = roc_auc_score(y_val, val_prob)
    cv_scores.append(fold_auc)
    
    # Accumulate Test Predictions across iterations
    test_preds += model.predict_proba(X_test)[:, 1] / skf.n_splits
    
    print(f"Fold {fold + 1} ROC-AUC: {fold_auc:.4f}")

print(f"\nOverall Out-of-Fold Mean ROC-AUC: {np.mean(cv_scores):.4f} (+/- {np.std(cv_scores):.4f})")

## 5. Generate Submission File
Finally, assemble the prediction vector mapping to test matrix tracking IDs.

In [ ]:
submission = pd.DataFrame({
    'node_id': test_df['node_id'],
    'fault_probability': test_preds
})

submission.to_csv('submission.csv', index=False)
print("Submission file 'submission.csv' generated successfully!")
submission.head()